# 🚦 Road Signs Detection — YOLOv8
### Classes: No Entry | Speed Bump | Pedestrian Crossing
---
**شغّلي الـ cells بالترتيب من فوق لتحت**

## Cell 1 — ربط Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive متوصل!')

## Cell 2 — تثبيت المكتبات

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()
print('ultralytics جاهزة!')

## Cell 3 — دمج الـ Dataset

In [ ]:
import shutil
from pathlib import Path

BASE_DIR   = Path('/content/drive/MyDrive/graduation_project_dataset')
OUTPUT_DIR = Path('/content/merged_dataset')

SOURCES = {
    0: ['no_entry1', 'no_entry2', 'no_entry3', 'no_entry4'],
    1: ['speed_bump1', 'speed_bump2', 'speed_bump3', 'speed_bump4', 'speed_bump5'],
    2: ['crosswalk'],
}

SPLITS = ['train', 'valid', 'test']

def fix_label(label_path, new_class_id):
    lines = label_path.read_text(encoding='utf-8').strip().splitlines()
    fixed = []
    for line in lines:
        parts = line.split()
        if len(parts) < 5:
            continue
        parts[0] = str(new_class_id)
        fixed.append(' '.join(parts))
    return '\n'.join(fixed)

counters = {s: {'img': 0, 'lbl': 0, 'missing': 0} for s in SPLITS}

for split in SPLITS:
    img_out = OUTPUT_DIR / split / 'images'
    lbl_out = OUTPUT_DIR / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for cls_id, folders in SOURCES.items():
        for folder in folders:
            img_src = BASE_DIR / folder / split / 'images'
            lbl_src = BASE_DIR / folder / split / 'labels'

            if not img_src.exists():
                print(f'  [SKIP] {folder}/{split} — غير موجود')
                continue

            images = sorted(img_src.glob('*.jpg')) + sorted(img_src.glob('*.png'))
            print(f'  [OK]  {folder}/{split} → class {cls_id} | {len(images)} صورة')

            for img_path in images:
                new_stem = f'c{cls_id}_{folder}_{img_path.stem}'
                shutil.copy2(img_path, img_out / (new_stem + img_path.suffix))
                counters[split]['img'] += 1

                lbl_path = lbl_src / (img_path.stem + '.txt')
                if lbl_path.exists():
                    (lbl_out / (new_stem + '.txt')).write_text(
                        fix_label(lbl_path, cls_id), encoding='utf-8')
                    counters[split]['lbl'] += 1
                else:
                    counters[split]['missing'] += 1

print('\n========= ملخص الدمج =========')
for split, c in counters.items():
    print(f'  {split:6s} | صور: {c["img"]:4d} | labels: {c["lbl"]:4d} | ناقص: {c["missing"]}')
print('================================')
print('الدمج خلص!')

## Cell 4 — كتابة data.yaml

In [ ]:
yaml_content = """train: /content/merged_dataset/train/images
val:   /content/merged_dataset/valid/images

nc: 3
names:
  0: no_entry
  1: speed_bump
  2: pedestrian_crossing
"""

yaml_path = Path('/content/merged_dataset/data.yaml')
yaml_path.write_text(yaml_content, encoding='utf-8')

print('data.yaml كُتب بنجاح!')
print(yaml_content)

## Cell 5 — التحقق من صحة الـ Dataset

In [ ]:
from pathlib import Path

DATASET_DIR  = Path('/content/merged_dataset')
VALID_CLASSES = {0, 1, 2}
SPLITS        = ['train', 'valid', 'test']
NAMES         = {0: 'no_entry', 1: 'speed_bump', 2: 'pedestrian_crossing'}

all_ok      = True
class_count = {0: 0, 1: 0, 2: 0}
total_img   = 0

for split in SPLITS:
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    if not img_dir.exists():
        continue

    images = sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png'))
    print(f'\n--- {split} ({len(images)} صورة) ---')
    total_img += len(images)
    errors = 0

    for img_path in images:
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            print(f'  [ERROR] مفيش label: {img_path.name}')
            all_ok = False
            errors += 1
            continue

        lines = lbl_path.read_text().strip().splitlines()
        if not lines:
            print(f'  [ERROR] label فاضي: {lbl_path.name}')
            all_ok = False
            errors += 1
            continue

        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                print(f'  [ERROR] format غلط: {lbl_path.name}')
                all_ok = False
                errors += 1
                break
            cls_id = int(parts[0])
            if cls_id not in VALID_CLASSES:
                print(f'  [ERROR] class ID غلط ({cls_id}): {lbl_path.name}')
                all_ok = False
                errors += 1
                break
            class_count[cls_id] += 1

    print(f'  errors: {errors}')

print('\n========= إحصائيات =========')
print(f'  إجمالي صور: {total_img}')
print('  توزيع الكلاسات:')
for cid, cnt in class_count.items():
    bar = '█' * (cnt // 100)
    print(f'    [{cid}] {NAMES[cid]:22s}: {cnt:5d}  {bar}')
print('============================')
if all_ok:
    print('  كل شي تمام! شغّلي Cell 6 هلق')
else:
    print('  في مشاكل — راجعي الـ errors فوق')

## Cell 6 — التدريب

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/merged_dataset/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,          # كولاب GPU يتحمل 16
    device=0,          # GPU الكولاب
    workers=2,

    # optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    # augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,

    # control
    patience=20,
    save=True,
    plots=True,
    val=True,

    project='/content/drive/MyDrive/graduation_project_results',
    name='road_signs_v1',
)

print('التدريب خلص!')

## Cell 7 — التقييم (Evaluation)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

best_model_path = '/content/drive/MyDrive/graduation_project_results/road_signs_v1/weights/best.pt'
model = YOLO(best_model_path)

metrics = model.val(data='/content/merged_dataset/data.yaml')

print('\n========= نتائج التقييم =========')
print(f'  mAP50     : {metrics.box.map50:.4f}')
print(f'  mAP50-95  : {metrics.box.map:.4f}')
print(f'  Precision : {metrics.box.mp:.4f}')
print(f'  Recall    : {metrics.box.mr:.4f}')
print('=================================')

## Cell 8 — عرض نتائج التدريب (Plots)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

results_dir = Path('/content/drive/MyDrive/graduation_project_results/road_signs_v1')

plots = [
    ('results.png',          'Training Curves'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('val_batch0_pred.jpg',  'Validation Predictions'),
    ('PR_curve.png',         'PR Curve'),
]

for filename, title in plots:
    path = results_dir / filename
    if path.exists():
        print(f'\n--- {title} ---')
        display(Image(str(path), width=700))
    else:
        print(f'[SKIP] {filename} غير موجود')

## Cell 9 — اختبار على صورة جديدة

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import shutil

# غيري المسار لأي صورة بدك تختبريها
TEST_IMAGE = '/content/merged_dataset/test/images'  # بياخد أول صورة من الـ test

from pathlib import Path
test_images = list(Path(TEST_IMAGE).glob('*.jpg'))[:3]

model = YOLO('/content/drive/MyDrive/graduation_project_results/road_signs_v1/weights/best.pt')

for img_path in test_images:
    results = model.predict(
        source=str(img_path),
        conf=0.25,
        save=True,
        project='/content/predictions',
        name='test',
        exist_ok=True,
    )
    print(f'\nصورة: {img_path.name}')
    for r in results:
        for box in r.boxes:
            cls  = int(box.cls)
            conf = float(box.conf)
            names = {0: 'no_entry', 1: 'speed_bump', 2: 'pedestrian_crossing'}
            print(f'  → {names[cls]} | confidence: {conf:.2f}')

# عرض الصور مع الـ bounding boxes
pred_dir = Path('/content/predictions/test')
for img in list(pred_dir.glob('*.jpg'))[:3]:
    display(Image(str(img), width=600))

## Cell 10 — حفظ الموديل النهائي على Drive

In [ ]:
import shutil
from pathlib import Path

src  = Path('/content/drive/MyDrive/graduation_project_results/road_signs_v1/weights/best.pt')
dest = Path('/content/drive/MyDrive/graduation_project_results/final_model.pt')

if src.exists():
    shutil.copy2(src, dest)
    print(f'الموديل النهائي محفوظ في Drive:')
    print(f'  {dest}')
    size_mb = dest.stat().st_size / 1024 / 1024
    print(f'  الحجم: {size_mb:.1f} MB')
else:
    print('[ERROR] best.pt')